In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [3]:
len(documents)

72

In [12]:
doc = documents[0:3]
doc

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [9]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [10]:
from dotenv import load_dotenv
load_dotenv()

import os
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [11]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

config = types.GenerateContentConfig(
    system_instruction=data_gen_instructions,
    response_mime_type="application/json",
    response_schema=Questions,
)

In [12]:
from evaluation_utils import llm_structured_retry
import json

In [13]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [21]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [26]:
avg_input_tokens = sum(u.prompt_token_count for u in usages) / len(usages)
print(avg_input_tokens)

1449.6666666666667


In [14]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")

In [21]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
df_ground_truth

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md
...,...,...
355,How should I break up a long article or transc...,07-project-example/lessons/07-chunking.md
356,"When I have several blog posts or wiki pages, ...",07-project-example/lessons/07-chunking.md
357,"For a single long PDF or video transcript, wha...",07-project-example/lessons/07-chunking.md
358,If I’m dealing with a book or other really lon...,07-project-example/lessons/07-chunking.md


Searching the chunks

In [3]:
import sys
import os

# Додаємо шлях до папки 02-vector-search, яка знаходиться на одному рівні з 04-evaluation
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
vector_search_dir = os.path.join(parent_dir, "02-vector-search")

if vector_search_dir not in sys.path:
    sys.path.append(vector_search_dir)

from embedder import Embedder

model_path = os.path.join(vector_search_dir, "models", "Xenova", "all-MiniLM-L6-v2")

embed = Embedder(path=model_path)

2026-07-13 20:42:17.005756273 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [6]:
from gitsource import chunk_documents
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)
X = np.array(embed.encode_batch([chunk["content"] for chunk in chunks]))

len(chunks)

295

In [7]:
from minsearch import Index

tindex = Index(text_fields=["content"])
tindex.fit(chunks)

def text_search(query, num_results=5):
    return tindex.search(query, num_results=num_results)

In [19]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

def vector_search(query, num_results=5):
    query_embedding = embed.encode(query)
    return vindex.search(query_embedding, num_results=num_results)

In [43]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [41]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

Q2. First result with text search

In [17]:
q = df_ground_truth["question"][0]
print(q)
text_search(q)[0]["filename"]

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


'01-agentic-rag/lessons/03-rag.md'

Q3. First result with vector search

In [47]:
q = df_ground_truth["question"][0]
print(q)
vector_search(q)[0]["filename"]

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


'01-agentic-rag/lessons/01-intro.md'

In [30]:
def compute_relevance_text(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [50]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [34]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [36]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)

Q4. Evaluating text search

In [ ]:
relevance_text = compute_relevance_total(ground_truth, text_search)

In [ ]:
hit_rate(relevance_text)

0.7583333333333333

Q5. Evaluating vector search

In [ ]:
relevance_vector = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [ ]:
mrr(relevance_vector)

0.5486111111111112

Q6. Tuning hybrid search

In [51]:
def mrr(relevance_total):
    scores = []
    for relevance in relevance_total:
        for rank, rel in enumerate(relevance):
            if rel:
                scores.append(1 / (rank + 1))
                break
        else:
            scores.append(0)
    return sum(scores) / len(scores)


for k in [1, 50, 100, 200]:
    search_fn = lambda query, k=k: hybrid_search(query, k=k)
    relevance_total = compute_relevance_total(ground_truth, search_fn)
    print(f"k={k}: MRR={mrr(relevance_total):.4f}")

  0%|          | 0/360 [00:00<?, ?it/s]

k=1: MRR=0.6482


  0%|          | 0/360 [00:00<?, ?it/s]

k=50: MRR=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=100: MRR=0.6379


  0%|          | 0/360 [00:00<?, ?it/s]

k=200: MRR=0.6379
